# Проверка контракта JSON

Ноутбук проверяет формат эталонов судебных актов. Вызов API YandexGPT на этом этапе не требуется.

`Extraction` описывает ответ модели, а `AnnotatedDocument` — сохранённый JSON с идентификатором исходного файла. Проверка схемы охватывает типы и ключи. Сверка фактов и цитат с исходным текстом выполняется отдельно.

In [ ]:
from pathlib import Path

from pydantic import ValidationError

from legal_ner.schema import AnnotatedDocument, Extraction

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
annotations_dir = project_dir / "data" / "annotations"

In [ ]:
paths = sorted(
    path
    for path in annotations_dir.glob("*.json")
    if path.name != "excluded_documents.json"
)
if not paths:
    raise FileNotFoundError(f"Нет эталонных JSON в {annotations_dir}")

errors = []
for path in paths:
    try:
        document = AnnotatedDocument.model_validate_json(path.read_text(encoding="utf-8"))
    except ValidationError as error:
        errors.append(f"{path.name}: {error}")
        continue
    if document.document_id != path.stem:
        errors.append(f"{path.name}: document_id={document.document_id!r}")
        continue
    print(f"{path.name}: OK")
    for index, party in enumerate(document.parties):
        if party.kpp is not None and len(party.kpp) != 9:
            print(f"  parties[{index}].kpp: в источнике {len(party.kpp)} цифр, ожидается 9")

if errors:
    raise ValueError("\n".join(errors))

print(f"Проверено эталонов: {len(paths)}")

In [ ]:
schema = Extraction.model_json_schema()
print("Обязательные поля ответа модели:", ", ".join(schema["required"]))